# Trace And Credit Window

Default execution recomputes reduced-budget live evidence for the device trace and delayed-credit window. Full-sweep cache mode is opt-in.

### Setup and Dependencies
Imports the trace package, plotting utilities, and configures the default execution mode.


In [ ]:
import os, math, json, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.stats import bootstrap_ci

RESULT_MODE = "live"
if RESULT_MODE not in {"live", "full_sweep_cache"}:
    raise ValueError("RESULT_MODE must be 'live' or 'full_sweep_cache'")

GREEN, INDIGO, RED, GOLD, GREY, PURPLE, INK = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2", "#b07cc6", "#2b2b2b"
VIR = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(True, color="0.88", lw=0.5)

def _ci_tuple(a):
    a = np.asarray(a, float)
    lo, hi = bootstrap_ci(a)
    return (float(a.mean()), float(a.std()), lo, hi)

def _cache(name):
    print(f"FULL-SWEEP CACHE: {paths.results_dir() / name}")
    return paths.load_result(name)

def _running(rw_2d, w=40):
    rw = np.asarray(rw_2d, float)
    w = max(2, min(w, rw.shape[1] - 1))
    cs = np.cumsum(np.insert(rw, 0, 0.0, axis=1), axis=1)
    rr = (cs[:, w:] - cs[:, :-w]) / w
    return rr.mean(0), np.percentile(rr, 2.5, axis=0), np.percentile(rr, 97.5, axis=0), w

print("RESULT_MODE:", RESULT_MODE)
print("data/results:", paths.results_dir())

from mrl_trace.device import TransientGate, tau_r
from mrl_trace.bandit import run_learning_and_window
from mrl_trace.maze import run_dmax_law
print(f"operating point V=1.5 -> fitted rise time tau_r = {tau_r(1.5):.2f} s")

def _series(name, y, min_len=2):
    arr = np.asarray(y, float).ravel()
    if arr.size < min_len or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has insufficient live data for plotting: n={arr.size}")
    return arr

def _values(name, y):
    arr = np.asarray(y, float).ravel()
    if arr.size == 0 or not np.isfinite(arr).any():
        raise RuntimeError(f"{name} has no finite live values for plotting")
    return arr

def _smooth(y, win=50):
    arr = _series("curve", y, min_len=2)
    win = int(win)
    if arr.size < max(5, win):
        return arr
    left = win // 2
    right = win - 1 - left
    padded = np.pad(arr, (left, right), mode="edge")
    kernel = np.ones(win, dtype=float) / float(win)
    return np.convolve(padded, kernel, mode="valid")


### Device Trace Visualization
Computes and plots the deterministic mathematical device trace based on the fitted device parameters.


In [ ]:
t = np.arange(0.0, 80.0, 0.05)
coincidence_at, coincidence_dur = 1.0, 0.5
taus = [1.0, 5.0, 20.0]
fig, ax = plt.subplots(figsize=(7.0, 3.8))
ax.axvspan(coincidence_at, coincidence_at + coincidence_dur, color=GOLD, alpha=0.30, lw=0)
ax.text(coincidence_at + coincidence_dur / 2, 1.06, "coincidence", color=INK, fontsize=8, ha="center")
for tl, col in zip(taus, VIR):
    g = TransientGate(V=1.5, tau_leak=tl, dt=0.05)
    e = g.trace(t, coincidence_at=coincidence_at, coincidence_dur=coincidence_dur)
    ax.plot(t, e, color=col, lw=2.0, label=rf"$\tau_{{leak}}={tl:g}$ s")
    ax.fill_between(t, 0, e, color=col, alpha=0.10, lw=0)
ax.set_xlabel("time after coincidence (s)"); ax.set_ylabel("eligibility trace e(t)")
ax.set_xlim(0, 80); ax.set_ylim(0, 1.12)
ax.set_title("Device transient: measured rise law with retention-set decay")
ax.legend(frameon=False, fontsize=9); _clean(ax); plt.show()
for tl in taus:
    e = TransientGate(V=1.5, tau_leak=tl, dt=0.05).trace(t, coincidence_at=coincidence_at, coincidence_dur=coincidence_dur)
    pk = t[int(np.argmax(e))]
    tp, ep = t[t > pk], e[t > pk]
    d_half = (tp[int(np.argmin(np.abs(ep - 0.5)))] - pk) if (ep < 0.5).any() else float("nan")
    print(f"tau_leak={tl:4.1f} s: peak @ {pk:4.1f} s, half-peak decay ~ {d_half:5.1f} s")

### Delayed Credit Window
Computes the delayed-credit window limits across different retention times on the associative bandit task.


In [ ]:
if RESULT_MODE == "live":
    r = run_learning_and_window(seeds=4, trials=400, delays=(1, 2, 5), tau_leaks=(10.0, 2.0, 0.5))
    src = "LIVE reduced: 4 seeds, 400 trials, 3 delays x 3 retentions"
else:
    r = _cache("tier3_results.npy")
    src = "full-sweep cache"

delays = np.asarray(r["delays"], float)
taus = sorted(r["reward_rate"].keys(), reverse=True)
fig, ax = plt.subplots(figsize=(5.0, 3.8))
for tl, col in zip(taus, VIR):
    y = np.asarray(r["reward_rate"][tl], float)
    ci = r["reward_rate_ci"][tl]
    lo = np.array([c[0] for c in ci]); hi = np.array([c[1] for c in ci])
    ax.plot(delays, y, "-o", ms=4, lw=1.7, color=col, label=rf"$\tau_{{leak}}={tl:g}$ s")
    ax.fill_between(delays, lo, hi, color=col, alpha=0.18, lw=0)
ax.axhline(r.get("crit", 0.75), ls="--", color=GREY, lw=1.0)
ax.axhline(0.5, ls=":", color=RED, lw=1.0)
ax.set_xscale("log"); ax.set_xticks(delays); ax.set_xticklabels([f"{d:g}" for d in delays])
ax.set_xlabel("action-to-reward delay D (s)"); ax.set_ylabel("final reward rate")
ax.set_ylim(0.35, 1.03); ax.set_title("Delayed credit window")
ax.legend(fontsize=8, frameon=False); _clean(ax)
for arr, col, lab in [(r["curve_device"], GREEN, "device trace"), (r["curve_notrace"], GREY, "no trace")]:
    m, lo, hi, w = _running(arr)
    x = np.arange(w, w + len(m))
fig.suptitle(f"Eligibility-trace learning diagnostic [{src}]", fontsize=11)
fig.tight_layout(); plt.show()
print("claim status: live-backed" if RESULT_MODE == "live" else "claim status: full-sweep cache")
print("max learnable delay per retention:", {f"{k:g}": v for k, v in r["max_learn"].items()})

### Retention-Delay Scaling Law
Evaluates the scaling capability mapping the device's $\tau_{leak}$ to the maximum learnable delay.


In [ ]:
if RESULT_MODE == "live":
    r = run_dmax_law(
        seeds=2,
        episodes=180,
        taus=(0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0, 12.0),
        delays=(1, 2, 4, 8, 16, 32, 64, 128),
    )
    src = "LIVE reduced: 2 seeds, 180 episodes, 9 retentions x 8 delays"
else:
    r = _cache("exp8_dmax_law.npy")
    src = "full-sweep cache"

taus = np.asarray(r["taus"], float)
dvals = np.array([r["dmax"][t] for t in r["taus"]], float)
censored = np.asarray(r.get("censored", [False] * len(taus)), bool)
resolved = (~censored) & np.isfinite(dvals) & (dvals > 0)
monotone = bool(np.sum(resolved) < 2 or np.all(np.diff(dvals[resolved]) >= -1e-9))
right_censored = bool(censored.any())
k, r2 = float(r.get("k", np.nan)), float(r.get("r2", np.nan))

fig, ax = plt.subplots(figsize=(6.6, 4.2))
if np.sum(resolved) >= 2:
    ax.plot(taus[resolved], dvals[resolved], "-o", color=GREEN, lw=1.8, ms=4.5, label="resolved Dmax")
    if np.isfinite(k):
        xx = np.linspace(0, taus.max() * 1.05, 160)
        ax.plot(xx, k * xx, color=INK, lw=1.2, ls="--", label=rf"origin fit $D_{{max}}={k:.1f}\tau$")
else:
    ax.scatter(taus[resolved], dvals[resolved], s=58, color=GREEN, edgecolor="white", zorder=4, label="resolved Dmax")
if censored.any():
    ax.scatter(taus[censored], dvals[censored], s=66, marker="^", color=GOLD, edgecolor="white", zorder=5, label="right-censored")
ax.set_xlabel("retention tau_leak (s)"); ax.set_ylabel("max learnable delay Dmax (s)")
ax.set_title(f"Retention-delay scaling [{src}]")
ax.legend(frameon=False, fontsize=8); _clean(ax); plt.show()
print("Dmax orientation:", {
    "monotone_resolved": monotone,
    "right_censored": right_censored,
    "k_origin": round(k, 3) if np.isfinite(k) else None,
    "r2": round(r2, 3) if np.isfinite(r2) else None,
    "values": {float(t): round(float(r['dmax'][t]), 2) for t in r['taus']},
})
# Full-scale regeneration:
# python -m mrl_trace.maze --exp8 --full